# Analyze Account Data Sources

We have three sources for account and account holder data:
1. Direct download of the account data
2. Download through the power BI app
3. Inferred from transaction data

We need to examine what are the differences in different source and which information
we take from which of the sources.

## Packages and options

In [17]:
# add the parent directory to the sys.path
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd

In [18]:
fn_direct = "../data/source/automatic/eutl_accounts.csv"
fn_bi = "../data/source/manual/accounts.xlsx"
fn_trans = "../data/source/automatic/eutl_transactions.csv"

## Get data

Account data from direct download:

In [19]:
df_acc_direct = (
    pd.read_csv(fn_direct)
    .assign(
        account_id=lambda df: df["REGISTRY_CODE"] + "_" + df["ACCOUNT_IDENTIFIER"].astype(str),
    )
)
df_acc_direct.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47683 entries, 0 to 47682
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   ACCOUNT_IDENTIFIER    47683 non-null  int64 
 1   REGISTRY_CODE         47683 non-null  object
 2   REGISTRY_NAME         47683 non-null  object
 3   ACCOUNT_NAME          47683 non-null  object
 4   ACCOUNT_TYPE          47683 non-null  object
 5   ETS_ACCOUNT_TYPE      28365 non-null  object
 6   FULL_TYPE             47654 non-null  object
 7   OPEN_DATE             47647 non-null  object
 8   END_OF_VALIDITY_DATE  47683 non-null  object
 9   IS_CLOSURE_PENDING    47683 non-null  object
 10  SNAPSHOT_DATE         47683 non-null  object
 11  account_id            47683 non-null  object
dtypes: int64(1), object(11)
memory usage: 4.4+ MB


Power BI data

In [20]:
df_acc_bi = (
    pd.read_excel(fn_bi, skipfooter=2)
    .rename(columns={"..1": "registry_id"})
    .drop(columns=".")
    .assign(
        account_id=lambda df: df["registry_id"] + "_" + df["Account Identifier"].astype(str),
    )
)
df_acc_bi.info()

/Users/jan/git/eutl_scraper_v2/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47389 entries, 0 to 47388
Data columns (total 15 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   Account Identifier                                   47389 non-null  int64  
 1   National Administrator                               47389 non-null  object 
 2   Account Type                                         47389 non-null  object 
 3   Account Holder Name                                  47389 non-null  object 
 4   Account Name                                         47389 non-null  object 
 5   Installation/Aircraft Operator/Maritime Operator ID  22158 non-null  float64
 6   Company Registration No                              42781 non-null  object 
 7   Main Address Line                                    47380 non-null  object 
 8   City                                                 47380 non-nul

Transaction data

In [22]:
df_acc_trans = pd.read_csv("../data/source/automatic/eutl_transactions.csv")
df_acc_trans.info()

/var/folders/w2/k3rfwxh15jv4m5ttwmmc55j00000gn/T/ipykernel_90608/2211376138.py:1: DtypeWarning: Columns (20,22,23,24,25,26,27,28,29,47,49,50,51,52,53,54,55,56,60,62,65) have mixed types. Specify dtype option on import or set low_memory=False.
  df_acc_trans = pd.read_csv("../data/source/automatic/eutl_transactions.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2142475 entries, 0 to 2142474
Data columns (total 67 columns):
 #   Column                                                   Dtype  
---  ------                                                   -----  
 0   TRANSACTION_ID                                           object 
 1   TRANSACTION_TYPE                                         object 
 2   TRANSACTION_DATE                                         object 
 3   TRANSACTION_STATUS                                       object 
 4   TRANSFERRING_REGISTRY_NAME                               object 
 5   TRANSFERRING_ACCOUNT_TYPE1                               float64
 6   TRANSFERRING_ACCOUNT_TYPE2                               object 
 7   TRANSFERRING_ACCOUNT_TYPE3                               object 
 8   TRANSFERRING_ACCOUNT_OPEN_DT                             object 
 9   TRANSFERRING_ACCOUNT_END_OF_VALIDITY                     object 
 10  TRANSFERRING_ACCOUNT_NAME                 

## How do direct downloads compare against power bi accounts?

Overall we observe:

1. Only direct downloads provide information on dates
2. Only the PowerBi downloads provide information on the account holder 

In [14]:
acc_only_direct = set(df_acc_direct["account_id"]) - set(df_acc_bi["account_id"])
acc_only_bi = set(df_acc_bi["account_id"]) - set(df_acc_direct["account_id"])
print(f"Accounts only in direct download: {len(acc_only_direct)}")
print(f"Accounts only in BI download: {len(acc_only_bi)}")

Accounts only in direct download: 294
Accounts only in BI download: 0


### Account types

Account types are aggregated by groups and more disaggregated in the Power BI app.

In [11]:
direct, bi = "FULL_TYPE", "Account Type"
df_ = df_acc[(df_acc[direct].str.strip() != df_acc[bi].str.strip())]
print(f"{len(df_)} differing account types between direct downloads and Power BI")
df_ = (
    df_[[direct, bi]]
    .sort_values(by=direct)
    .drop_duplicates()
)
df_


298 differing account types between direct downloads and Power BI


,FULL_TYPE,Account Type
14028,AEA Deletion Account,NaN
14027,AEA Total quantity Account,NaN
14137,ESD Compliance Account,NaN
36747,Former Operator Holding Account,NaN
39111,Maritime Operator Holding Account,NaN
3845,Operator Holding Account,NaN
2501,Party Holding Account,NaN
2488,Person Account in National Registry,NaN
2505,Retirement Account,NaN
10894,Trading Account,NaN


In [ ]:
to_check = {
    "Account Name": "ACCOUNT_NAME",
    
}

for right, left in to_check.items():
    df_ = df_acc[(df_acc[left].str.strip() != df_acc[right].str.strip())]
    if not df_.empty:
        df_ = df_[[left, right]].sort_values(by=left).drop_duplicates()
        break
df_acc[[left, right]].drop_duplicates().sort_values(by=left)
df_

,ACCOUNT_NAME,Account Name
1840,8160 MARI KOKAKO,NaN
13297,A/S GLOBAL RISK MANAGEMENT LTD. HOLDING,A/S Global Risk Management Ltd. Holding
28091,A2A S.p.A.,A2A S.P.A.
34787,A2A Trading,a2a Trading
22621,ABN AMRO BANK N.v.,ABN AMRO BANK N.V.
...,...,...
33248,stabilimento di Brindisi,Stabilimento di Brindisi
14257,ubr logistik,UBR Logistik
34803,veronagest,Veronagest
5644,vertus energiehandel gmbh,Vertus Energiehandel Gmbh
